In [0]:
from pyspark.sql import functions as F

CATALOG = "dbw_retail_lakehouse_dev_eas_001"

SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

In [0]:
df_dim_customers = (
    spark.table(f"{SILVER}.customers")
    .select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "country",
        "created_at"
    )
    .withColumn(
        "full_name",
        F.concat_ws(" ", "first_name", "last_name")
    )
)

display(df_dim_customers)

In [0]:
(
    df_dim_customers
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.dim_customers")
)

In [0]:
df_dim_products = (
    spark.table(f"{SILVER}.products")
    .select(
        "product_id",
        "product_name",
        "category",
        "unit_price",
        "is_active"
    )
)

display(df_dim_products)

In [0]:
(
    df_dim_products
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.dim_products")
)

In [0]:
df_fact_orders = (
    spark.table(f"{SILVER}.orders")
    .select(
        "order_id",
        "customer_id",
        "product_id",
        "quantity",
        "unit_price",
        "line_amount",
        "order_timestamp",
        "status"
    )
    .withColumn(
        "order_date",
        F.to_date("order_timestamp")
    )
)

display(df_fact_orders)

In [0]:
(
    df_fact_orders
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.fact_orders")
)

In [0]:
%sql

SHOW TABLES
IN dbw_retail_lakehouse_dev_eas_001.gold;

In [0]:
%sql

SELECT 'dim_customers' AS table_name, COUNT(*) AS row_count
FROM dbw_retail_lakehouse_dev_eas_001.gold.dim_customers

UNION ALL

SELECT 'dim_products', COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.gold.dim_products

UNION ALL

SELECT 'fact_orders', COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.gold.fact_orders;

In [0]:
df_customer_sales = (
    spark.table(f"{GOLD}.fact_orders").alias("f")
    .join(
        spark.table(f"{GOLD}.dim_customers").alias("c"),
        on="customer_id",
        how="left"
    )
    .groupBy(
        "customer_id",
        "full_name",
        "country"
    )
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("units_purchased"),
        F.sum("line_amount").alias("total_sales")
    )
    .orderBy(
        F.col("total_sales").desc()
    )
)

display(df_customer_sales)

In [0]:
(
    df_customer_sales
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.mart_customer_sales")
)

In [0]:
df_product_sales = (
    spark.table(f"{GOLD}.fact_orders").alias("f")
    .join(
        spark.table(f"{GOLD}.dim_products").alias("p"),
        on="product_id",
        how="left"
    )
    .groupBy(
        "product_id",
        "product_name",
        "category"
    )
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("units_sold"),
        F.sum("line_amount").alias("total_sales")
    )
    .orderBy(
        F.col("total_sales").desc()
    )
)

display(df_product_sales)

In [0]:
(
    df_product_sales
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.mart_product_sales")
)

In [0]:
df_daily_sales = (
    spark.table(f"{GOLD}.fact_orders")
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("units_sold"),
        F.sum("line_amount").alias("total_sales")
    )
    .orderBy("order_date")
)

display(df_daily_sales)

In [0]:
(
    df_daily_sales
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{GOLD}.mart_daily_sales")
)

In [0]:
%sql

SHOW TABLES
IN dbw_retail_lakehouse_dev_eas_001.gold;

In [0]:
%sql

SELECT 'dim_customers' AS table_name, COUNT(*) AS row_count
FROM dbw_retail_lakehouse_dev_eas_001.gold.dim_customers

UNION ALL

SELECT 'dim_products', COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.gold.dim_products

UNION ALL

SELECT 'fact_orders', COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.gold.fact_orders

UNION ALL

SELECT 'mart_customer_sales', COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.gold.mart_customer_sales

UNION ALL

SELECT 'mart_product_sales', COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.gold.mart_product_sales

UNION ALL

SELECT 'mart_daily_sales', COUNT(*)
FROM dbw_retail_lakehouse_dev_eas_001.gold.mart_daily_sales;